# Querying JSON: Filters, Aggregation, and Paths

## Overview
This notebook covers the full range of JSON query patterns: expanding arrays, filtering on nested values, aggregating across JSON elements, and using JSONPath expressions.

**Core query patterns:**
```sql
-- SQLite: expand array to rows
FROM table, json_each(col->'$.arr') AS jt
WHERE json_extract(jt.value, '$.flag') = 'H'

-- PostgreSQL: same pattern
FROM table, jsonb_array_elements(col->'arr') AS t
WHERE t->>'flag' = 'H'

-- PostgreSQL: filter + aggregate in one pass
SELECT AVG((t->>'value')::numeric)
FROM   lab_results, jsonb_array_elements(tests) AS t
WHERE  t->>'test' = 'HbA1c'
```

**Dataset:** lab results with a `tests` JSON array (multiple tests per visit), medications with a `details` JSON object, patient `clinical` with nested conditions and vitals arrays.

---

In [1]:
import sqlite3, json

conn = sqlite3.connect(':memory:')
conn.row_factory = sqlite3.Row

# Healthcare dataset: patients with JSON clinical data
conn.executescript("""
CREATE TABLE patients (
    patient_id   TEXT PRIMARY KEY,
    full_name    TEXT NOT NULL,
    demographics JSON NOT NULL,   -- age, dob, province, contact info
    clinical     JSON NOT NULL,   -- conditions, allergies, vitals history
    preferences  JSON             -- notification prefs, language, etc.
);

CREATE TABLE lab_results (
    lab_id      INTEGER PRIMARY KEY AUTOINCREMENT,
    patient_id  TEXT NOT NULL,
    result_date TEXT NOT NULL,
    tests       JSON NOT NULL    -- array of {test, value, unit, ref_range, flag}
);

CREATE TABLE medications (
    med_id      INTEGER PRIMARY KEY AUTOINCREMENT,
    patient_id  TEXT NOT NULL,
    prescribed  TEXT NOT NULL,
    details     JSON NOT NULL    -- drug, dose, frequency, prescriber, side_effects
);
""")

patients = [
  ('P001','Alice Ngata',
   json.dumps({'age':45,'dob':'1980-03-15','province':'NB','contact':{'phone':'506-555-0101','email':'alice@email.com'},'insurance':{'plan':'premium','id':'INS-00123'}}),
   json.dumps({'conditions':['Hypertension','Hyperlipidaemia'],'allergies':[],'vitals':[{'date':'2023-04-10','bp':'142/88','weight_kg':72},{'date':'2023-10-01','bp':'128/82','weight_kg':70}],'smoker':False}),
   json.dumps({'language':'en','notifications':{'email':True,'sms':False},'portal_enabled':True})),
  ('P002','Bob Chen',
   json.dumps({'age':52,'dob':'1972-07-22','province':'ON','contact':{'phone':'416-555-2233','email':'bob@email.com'},'insurance':{'plan':'standard','id':'INS-00456'}}),
   json.dumps({'conditions':['Hypertension','Type 2 Diabetes'],'allergies':['Penicillin'],'vitals':[{'date':'2023-05-15','bp':'148/92','weight_kg':88},{'date':'2024-01-10','bp':'138/86','weight_kg':86}],'smoker':True}),
   json.dumps({'language':'en','notifications':{'email':True,'sms':True},'portal_enabled':True})),
  ('P003','Fatima Rashid',
   json.dumps({'age':38,'dob':'1986-11-05','province':'BC','contact':{'phone':'604-555-9900','email':'fatima@email.com'},'insurance':{'plan':'premium','id':'INS-00789'}}),
   json.dumps({'conditions':['Asthma','Obesity'],'allergies':['Sulfa drugs','NSAIDs'],'vitals':[{'date':'2023-08-20','bp':'148/92','weight_kg':95}],'smoker':False}),
   json.dumps({'language':'fr','notifications':{'email':False,'sms':True},'portal_enabled':False})),
  ('P004','James MacLeod',
   json.dumps({'age':61,'dob':'1963-01-30','province':'NS','contact':{'phone':'902-555-7788','email':'james@email.com'},'insurance':{'plan':'standard','id':'INS-01011'}}),
   json.dumps({'conditions':['Type 2 Diabetes'],'allergies':[],'vitals':[{'date':'2023-09-01','bp':'118/76','weight_kg':80}],'smoker':False}),
   json.dumps({'language':'en','notifications':{'email':True,'sms':False},'portal_enabled':True})),
  ('P005','Mei-Ling Tran',
   json.dumps({'age':58,'dob':'1966-08-19','province':'QC','contact':{'phone':'514-555-1122','email':'mei@email.com'},'insurance':{'plan':'premium','id':'INS-01213'}}),
   json.dumps({'conditions':['Type 2 Diabetes','Hypertension','CKD Stage 3'],'allergies':['Metformin'],'vitals':[{'date':'2023-11-10','bp':'155/96','weight_kg':65},{'date':'2024-02-01','bp':'145/90','weight_kg':64}],'smoker':False}),
   json.dumps({'language':'fr','notifications':{'email':True,'sms':True},'portal_enabled':True})),
]
conn.executemany("INSERT INTO patients VALUES (?,?,?,?,?)", patients)

lab_rows = [
  ('P001','2023-10-01', json.dumps([
    {'test':'HbA1c','value':7.2,'unit':'%','ref_range':'<5.7','flag':'H'},
    {'test':'eGFR','value':68,'unit':'mL/min/1.73m2','ref_range':'>60','flag':'N'},
    {'test':'LDL','value':3.1,'unit':'mmol/L','ref_range':'<2.6','flag':'H'}])),
  ('P002','2024-01-10', json.dumps([
    {'test':'HbA1c','value':8.4,'unit':'%','ref_range':'<5.7','flag':'H'},
    {'test':'eGFR','value':55,'unit':'mL/min/1.73m2','ref_range':'>60','flag':'L'},
    {'test':'Creatinine','value':115,'unit':'umol/L','ref_range':'62-106','flag':'H'}])),
  ('P004','2023-09-01', json.dumps([
    {'test':'HbA1c','value':7.8,'unit':'%','ref_range':'<5.7','flag':'H'},
    {'test':'eGFR','value':72,'unit':'mL/min/1.73m2','ref_range':'>60','flag':'N'}])),
  ('P005','2024-02-01', json.dumps([
    {'test':'HbA1c','value':9.1,'unit':'%','ref_range':'<5.7','flag':'H'},
    {'test':'eGFR','value':38,'unit':'mL/min/1.73m2','ref_range':'>60','flag':'L'},
    {'test':'Potassium','value':5.8,'unit':'mmol/L','ref_range':'3.5-5.0','flag':'H'}])),
]
conn.executemany("INSERT INTO lab_results (patient_id,result_date,tests) VALUES (?,?,?)", lab_rows)

med_rows = [
  ('P001','2023-01-15', json.dumps({'drug':'Lisinopril','dose':'10mg','frequency':'once daily','prescriber':'Dr. Lee','side_effects':['dizziness','dry cough']})),
  ('P001','2023-01-15', json.dumps({'drug':'Atorvastatin','dose':'40mg','frequency':'once daily','prescriber':'Dr. Lee','side_effects':['myalgia']})),
  ('P002','2022-06-01', json.dumps({'drug':'Metformin','dose':'500mg','frequency':'BID','prescriber':'Dr. Pham','side_effects':['nausea','diarrhoea']})),
  ('P002','2022-06-01', json.dumps({'drug':'Lisinopril','dose':'5mg','frequency':'once daily','prescriber':'Dr. Pham','side_effects':[]})),
  ('P003','2021-09-10', json.dumps({'drug':'Salbutamol','dose':'2.5mg','frequency':'PRN','prescriber':'Dr. Singh','side_effects':['tremor','palpitations']})),
  ('P005','2023-03-01', json.dumps({'drug':'Insulin glargine','dose':'20 units','frequency':'nocte','prescriber':'Dr. Pham','side_effects':['hypoglycaemia']})),
]
conn.executemany("INSERT INTO medications (patient_id,prescribed,details) VALUES (?,?,?)", med_rows)
conn.commit()

print("Healthcare JSON dataset ready:")
for tbl in ("patients","lab_results","medications"):
    n = conn.execute(f"SELECT COUNT(*) FROM {tbl}").fetchone()[0]
    print(f"  {tbl}: {n} rows")

# Preview one JSON column
row = conn.execute("SELECT full_name, demographics FROM patients LIMIT 1").fetchone()
print(f"\nSample demographics JSON for {row['full_name']}:")
print(json.dumps(json.loads(row['demographics']), indent=2))


print("=== Querying deeply nested JSON: lab results array ===")
print()

# Each lab_results row has a tests JSON array: [{test, value, unit, ref_range, flag}, ...]
# Expand to one row per test using json_each
rows = conn.execute("""
    SELECT lr.patient_id,
           p.full_name,
           lr.result_date,
           jt.value AS test_json
    FROM   lab_results lr
    JOIN   patients p ON p.patient_id = lr.patient_id,
           json_each(lr.tests) AS jt
    ORDER  BY lr.patient_id, lr.result_date
""").fetchall()

print("Expanded lab tests (one row per test):")
print(f"  {'patient':<12s}  {'date':<12s}  {'test':<12s}  {'value':<8s}  {'flag'}")
print("  " + "-"*55)
for r in rows:
    t = json.loads(r['test_json'])
    print(f"  {r['patient_id']:<12s}  {r['result_date']:<12s}  {t['test']:<12s}  {str(t['value']):<8s}  {t['flag']}")

print()
# Filter on nested JSON field inside array elements
rows2 = conn.execute("""
    SELECT p.full_name,
           lr.result_date,
           json_extract(jt.value, '$.test')  AS test,
           json_extract(jt.value, '$.value') AS value,
           json_extract(jt.value, '$.flag')  AS flag
    FROM   lab_results lr
    JOIN   patients p ON p.patient_id = lr.patient_id,
           json_each(lr.tests) AS jt
    WHERE  json_extract(jt.value, '$.flag') = 'H'
       AND json_extract(jt.value, '$.test') = 'HbA1c'
    ORDER  BY CAST(json_extract(jt.value, '$.value') AS REAL) DESC
""").fetchall()
print("High-flagged HbA1c results (flag='H'), worst first:")
for r in rows2:
    print(f"  {r['full_name']:<22s}  {r['test']:<8s}  value={r['value']}  flag={r['flag']}")

print()
print("PostgreSQL equivalents:")
print("""
  -- Expand JSONB array to rows:
  SELECT p.full_name, lr.result_date,
         t->>'test'  AS test,
         t->>'value' AS value,
         t->>'flag'  AS flag
  FROM   lab_results lr
  JOIN   patients p ON p.patient_id = lr.patient_id,
         jsonb_array_elements(lr.tests) AS t
  WHERE  t->>'flag' = 'H'
    AND  t->>'test' = 'HbA1c'
  ORDER  BY (t->>'value')::numeric DESC;
""")


Healthcare JSON dataset ready:
  patients: 5 rows
  lab_results: 4 rows
  medications: 6 rows

Sample demographics JSON for Alice Ngata:
{
  "age": 45,
  "dob": "1980-03-15",
  "province": "NB",
  "contact": {
    "phone": "506-555-0101",
    "email": "alice@email.com"
  },
  "insurance": {
    "plan": "premium",
    "id": "INS-00123"
  }
}
=== Querying deeply nested JSON: lab results array ===

Expanded lab tests (one row per test):
  patient       date          test          value     flag
  -------------------------------------------------------
  P001          2023-10-01    HbA1c         7.2       H
  P001          2023-10-01    eGFR          68        N
  P001          2023-10-01    LDL           3.1       H
  P002          2024-01-10    HbA1c         8.4       H
  P002          2024-01-10    eGFR          55        L
  P002          2024-01-10    Creatinine    115       H
  P004          2023-09-01    HbA1c         7.8       H
  P004          2023-09-01    eGFR          72       

---
## Aggregation on JSON data

In [2]:
import sqlite3, json

conn = sqlite3.connect(':memory:')
conn.row_factory = sqlite3.Row

# Healthcare dataset: patients with JSON clinical data
conn.executescript("""
CREATE TABLE patients (
    patient_id   TEXT PRIMARY KEY,
    full_name    TEXT NOT NULL,
    demographics JSON NOT NULL,   -- age, dob, province, contact info
    clinical     JSON NOT NULL,   -- conditions, allergies, vitals history
    preferences  JSON             -- notification prefs, language, etc.
);

CREATE TABLE lab_results (
    lab_id      INTEGER PRIMARY KEY AUTOINCREMENT,
    patient_id  TEXT NOT NULL,
    result_date TEXT NOT NULL,
    tests       JSON NOT NULL    -- array of {test, value, unit, ref_range, flag}
);

CREATE TABLE medications (
    med_id      INTEGER PRIMARY KEY AUTOINCREMENT,
    patient_id  TEXT NOT NULL,
    prescribed  TEXT NOT NULL,
    details     JSON NOT NULL    -- drug, dose, frequency, prescriber, side_effects
);
""")

patients = [
  ('P001','Alice Ngata',
   json.dumps({'age':45,'dob':'1980-03-15','province':'NB','contact':{'phone':'506-555-0101','email':'alice@email.com'},'insurance':{'plan':'premium','id':'INS-00123'}}),
   json.dumps({'conditions':['Hypertension','Hyperlipidaemia'],'allergies':[],'vitals':[{'date':'2023-04-10','bp':'142/88','weight_kg':72},{'date':'2023-10-01','bp':'128/82','weight_kg':70}],'smoker':False}),
   json.dumps({'language':'en','notifications':{'email':True,'sms':False},'portal_enabled':True})),
  ('P002','Bob Chen',
   json.dumps({'age':52,'dob':'1972-07-22','province':'ON','contact':{'phone':'416-555-2233','email':'bob@email.com'},'insurance':{'plan':'standard','id':'INS-00456'}}),
   json.dumps({'conditions':['Hypertension','Type 2 Diabetes'],'allergies':['Penicillin'],'vitals':[{'date':'2023-05-15','bp':'148/92','weight_kg':88},{'date':'2024-01-10','bp':'138/86','weight_kg':86}],'smoker':True}),
   json.dumps({'language':'en','notifications':{'email':True,'sms':True},'portal_enabled':True})),
  ('P003','Fatima Rashid',
   json.dumps({'age':38,'dob':'1986-11-05','province':'BC','contact':{'phone':'604-555-9900','email':'fatima@email.com'},'insurance':{'plan':'premium','id':'INS-00789'}}),
   json.dumps({'conditions':['Asthma','Obesity'],'allergies':['Sulfa drugs','NSAIDs'],'vitals':[{'date':'2023-08-20','bp':'148/92','weight_kg':95}],'smoker':False}),
   json.dumps({'language':'fr','notifications':{'email':False,'sms':True},'portal_enabled':False})),
  ('P004','James MacLeod',
   json.dumps({'age':61,'dob':'1963-01-30','province':'NS','contact':{'phone':'902-555-7788','email':'james@email.com'},'insurance':{'plan':'standard','id':'INS-01011'}}),
   json.dumps({'conditions':['Type 2 Diabetes'],'allergies':[],'vitals':[{'date':'2023-09-01','bp':'118/76','weight_kg':80}],'smoker':False}),
   json.dumps({'language':'en','notifications':{'email':True,'sms':False},'portal_enabled':True})),
  ('P005','Mei-Ling Tran',
   json.dumps({'age':58,'dob':'1966-08-19','province':'QC','contact':{'phone':'514-555-1122','email':'mei@email.com'},'insurance':{'plan':'premium','id':'INS-01213'}}),
   json.dumps({'conditions':['Type 2 Diabetes','Hypertension','CKD Stage 3'],'allergies':['Metformin'],'vitals':[{'date':'2023-11-10','bp':'155/96','weight_kg':65},{'date':'2024-02-01','bp':'145/90','weight_kg':64}],'smoker':False}),
   json.dumps({'language':'fr','notifications':{'email':True,'sms':True},'portal_enabled':True})),
]
conn.executemany("INSERT INTO patients VALUES (?,?,?,?,?)", patients)

lab_rows = [
  ('P001','2023-10-01', json.dumps([
    {'test':'HbA1c','value':7.2,'unit':'%','ref_range':'<5.7','flag':'H'},
    {'test':'eGFR','value':68,'unit':'mL/min/1.73m2','ref_range':'>60','flag':'N'},
    {'test':'LDL','value':3.1,'unit':'mmol/L','ref_range':'<2.6','flag':'H'}])),
  ('P002','2024-01-10', json.dumps([
    {'test':'HbA1c','value':8.4,'unit':'%','ref_range':'<5.7','flag':'H'},
    {'test':'eGFR','value':55,'unit':'mL/min/1.73m2','ref_range':'>60','flag':'L'},
    {'test':'Creatinine','value':115,'unit':'umol/L','ref_range':'62-106','flag':'H'}])),
  ('P004','2023-09-01', json.dumps([
    {'test':'HbA1c','value':7.8,'unit':'%','ref_range':'<5.7','flag':'H'},
    {'test':'eGFR','value':72,'unit':'mL/min/1.73m2','ref_range':'>60','flag':'N'}])),
  ('P005','2024-02-01', json.dumps([
    {'test':'HbA1c','value':9.1,'unit':'%','ref_range':'<5.7','flag':'H'},
    {'test':'eGFR','value':38,'unit':'mL/min/1.73m2','ref_range':'>60','flag':'L'},
    {'test':'Potassium','value':5.8,'unit':'mmol/L','ref_range':'3.5-5.0','flag':'H'}])),
]
conn.executemany("INSERT INTO lab_results (patient_id,result_date,tests) VALUES (?,?,?)", lab_rows)

med_rows = [
  ('P001','2023-01-15', json.dumps({'drug':'Lisinopril','dose':'10mg','frequency':'once daily','prescriber':'Dr. Lee','side_effects':['dizziness','dry cough']})),
  ('P001','2023-01-15', json.dumps({'drug':'Atorvastatin','dose':'40mg','frequency':'once daily','prescriber':'Dr. Lee','side_effects':['myalgia']})),
  ('P002','2022-06-01', json.dumps({'drug':'Metformin','dose':'500mg','frequency':'BID','prescriber':'Dr. Pham','side_effects':['nausea','diarrhoea']})),
  ('P002','2022-06-01', json.dumps({'drug':'Lisinopril','dose':'5mg','frequency':'once daily','prescriber':'Dr. Pham','side_effects':[]})),
  ('P003','2021-09-10', json.dumps({'drug':'Salbutamol','dose':'2.5mg','frequency':'PRN','prescriber':'Dr. Singh','side_effects':['tremor','palpitations']})),
  ('P005','2023-03-01', json.dumps({'drug':'Insulin glargine','dose':'20 units','frequency':'nocte','prescriber':'Dr. Pham','side_effects':['hypoglycaemia']})),
]
conn.executemany("INSERT INTO medications (patient_id,prescribed,details) VALUES (?,?,?)", med_rows)
conn.commit()

print("Healthcare JSON dataset ready:")
for tbl in ("patients","lab_results","medications"):
    n = conn.execute(f"SELECT COUNT(*) FROM {tbl}").fetchone()[0]
    print(f"  {tbl}: {n} rows")

# Preview one JSON column
row = conn.execute("SELECT full_name, demographics FROM patients LIMIT 1").fetchone()
print(f"\nSample demographics JSON for {row['full_name']}:")
print(json.dumps(json.loads(row['demographics']), indent=2))


print("=== Aggregation on JSON data ===")
print()

# Average HbA1c across all patients (extracted from JSON array)
rows = conn.execute("""
    SELECT AVG(CAST(json_extract(jt.value, '$.value') AS REAL)) AS avg_hba1c,
           COUNT(*)                                             AS n_tests
    FROM   lab_results lr,
           json_each(lr.tests) AS jt
    WHERE  json_extract(jt.value, '$.test') = 'HbA1c'
""").fetchone()
print(f"Average HbA1c across all patients: {rows['avg_hba1c']:.2f}% (n={rows['n_tests']})")

print()
# Per-patient: count conditions from JSON array
rows2 = conn.execute("""
    SELECT full_name,
           json_array_length(json_extract(clinical, '$.conditions')) AS n_conditions,
           json_extract(clinical, '$.conditions')                    AS conditions_json
    FROM   patients
    ORDER  BY n_conditions DESC
""").fetchall()
print("Condition count per patient:")
for r in rows2:
    conditions = json.loads(r['conditions_json'])
    print(f"  {r['full_name']:<22s}  n={r['n_conditions']}  {conditions}")

print()
# Most common condition across all patients
rows3 = conn.execute("""
    SELECT jc.value AS condition,
           COUNT(*) AS n_patients,
           GROUP_CONCAT(p.full_name, ', ') AS patients
    FROM   patients p,
           json_each(json_extract(p.clinical, '$.conditions')) AS jc
    GROUP  BY jc.value
    ORDER  BY n_patients DESC
""").fetchall()
print("Condition prevalence (all patients):")
for r in rows3:
    print(f"  {r['condition']:<28s}  n={r['n_patients']}  [{r['patients']}]")

print()
print("PostgreSQL aggregation on JSONB:")
print("""
  -- Average HbA1c from JSONB array:
  SELECT AVG((t->>'value')::numeric) AS avg_hba1c
  FROM   lab_results,
         jsonb_array_elements(tests) AS t
  WHERE  t->>'test' = 'HbA1c';

  -- Condition counts using json_agg for output:
  SELECT jc.value AS condition, COUNT(*) AS n_patients,
         json_agg(p.full_name) AS patient_names
  FROM   patients p,
         jsonb_array_elements_text(p.clinical->'conditions') AS jc(value)
  GROUP  BY jc.value
  ORDER  BY n_patients DESC;
""")


Healthcare JSON dataset ready:
  patients: 5 rows
  lab_results: 4 rows
  medications: 6 rows

Sample demographics JSON for Alice Ngata:
{
  "age": 45,
  "dob": "1980-03-15",
  "province": "NB",
  "contact": {
    "phone": "506-555-0101",
    "email": "alice@email.com"
  },
  "insurance": {
    "plan": "premium",
    "id": "INS-00123"
  }
}
=== Aggregation on JSON data ===

Average HbA1c across all patients: 8.12% (n=4)

Condition count per patient:
  Mei-Ling Tran           n=3  ['Type 2 Diabetes', 'Hypertension', 'CKD Stage 3']
  Alice Ngata             n=2  ['Hypertension', 'Hyperlipidaemia']
  Bob Chen                n=2  ['Hypertension', 'Type 2 Diabetes']
  Fatima Rashid           n=2  ['Asthma', 'Obesity']
  James MacLeod           n=1  ['Type 2 Diabetes']

Condition prevalence (all patients):
  Type 2 Diabetes               n=3  [Bob Chen, James MacLeod, Mei-Ling Tran]
  Hypertension                  n=3  [Alice Ngata, Bob Chen, Mei-Ling Tran]
  Obesity                       n=

---
## JSONPath expressions (PostgreSQL 12+)

In [3]:
print("=== jsonb_path_query: JSONPath expressions (PostgreSQL 12+) ===")
print()

print("JSONPath syntax (PostgreSQL 12+):")
jsonpath_examples = [
    ("$.age",                    "Top-level field"),
    ("$.contact.phone",          "Nested field"),
    ("$.vitals[0]",              "First array element"),
    ("$.vitals[*].bp",           "All bp values from vitals array"),
    ("$.vitals[*] ? (@.date > \"2023-06-01\")", "Array elements filtered by condition"),
    ("$.conditions[*] ? (@ == \"Hypertension\")", "Array element that equals a value"),
    ("strict $.missing",         "strict mode: error on missing path (default is lax)"),
    ("lax $.missing",            "lax mode: returns empty on missing path"),
]
print(f"  {'JSONPath':<46s}  Meaning")
print("  " + "-"*70)
for path, meaning in jsonpath_examples:
    print(f"  {path:<46s}  {meaning}")

print()
print("PostgreSQL JSONPath functions:")
pg_jsonpath = [
    ("jsonb_path_query(col, path)",        "Returns set of JSONB values matching path"),
    ("jsonb_path_query_first(col, path)",  "Returns first match (or NULL)"),
    ("jsonb_path_exists(col, path)",       "Returns boolean: does path match anything?"),
    ("jsonb_path_match(col, path)",        "Returns boolean result of path predicate"),
    ("jsonb_path_query_array(col, path)",  "Returns all matches as a JSONB array"),
]
for fn, desc in pg_jsonpath:
    print(f"  {fn:<44s}  {desc}")

print()
print("""PostgreSQL JSONPath examples:

  -- Get all bp readings from vitals array:
  SELECT full_name,
         jsonb_path_query_array(clinical, '$.vitals[*].bp') AS all_bp
  FROM   patients;

  -- Find patients whose most recent bp was high (systolic > 140):
  -- (requires extracting numeric part -- complex with pure JSONPath)
  SELECT full_name
  FROM   patients
  WHERE  jsonb_path_exists(clinical, '$.vitals[*] ? (@.bp starts with "14" || @.bp starts with "15")');

  -- Get all conditions containing 'Diabetes':
  SELECT full_name,
         jsonb_path_query(clinical, '$.conditions[*] ? (@ like_regex "Diabetes")')
  FROM   patients;
""")


=== jsonb_path_query: JSONPath expressions (PostgreSQL 12+) ===

JSONPath syntax (PostgreSQL 12+):
  JSONPath                                        Meaning
  ----------------------------------------------------------------------
  $.age                                           Top-level field
  $.contact.phone                                 Nested field
  $.vitals[0]                                     First array element
  $.vitals[*].bp                                  All bp values from vitals array
  $.vitals[*] ? (@.date > "2023-06-01")           Array elements filtered by condition
  $.conditions[*] ? (@ == "Hypertension")         Array element that equals a value
  strict $.missing                                strict mode: error on missing path (default is lax)
  lax $.missing                                   lax mode: returns empty on missing path

PostgreSQL JSONPath functions:
  jsonb_path_query(col, path)                   Returns set of JSONB values matching path
  js

---
## Common Pitfalls

**1. Not casting JSON numeric values before comparison**
`WHERE (lab_result->>'value') > 8.0` compares text lexicographically: `'9' > '80'` is false because `'9'` < `'8'` alphabetically. Cast first: `WHERE (lab_result->>'value')::numeric > 8.0`.

**2. Cross join with json_each producing wrong row counts**
`FROM patients, json_each(clinical->'conditions')` is a cross join (lateral). If a patient has 3 conditions, they appear 3 times. Forgetting this in aggregate queries causes inflated counts. Use `GROUP BY patient_id` to collapse back to one row per patient.

**3. jsonb_array_elements on a non-array**
`jsonb_array_elements(col->'conditions')` raises `cannot call jsonb_array_elements on a scalar` if `conditions` is a string, not an array. Validate with `WHERE jsonb_typeof(col->'conditions') = 'array'` before expanding.

**4. Using jsonb_path_query on PostgreSQL < 12**
JSONPath (`jsonb_path_query`, `jsonb_path_exists`) requires PostgreSQL 12+. Check `SHOW server_version` before using these functions. Use `@>` and `->>` as fallbacks for older versions.

**5. Slow aggregation without pre-filtering**
Aggregating `json_each(tests)` across all lab_results rows when you only need HbA1c evaluates every test element in every row. Add a pre-filter at the table level where possible, or use a generated column for frequently-queried JSON values.

**6. Forgetting that json_each key is a string index for arrays**
When expanding a JSON array with `json_each()` in SQLite, `key` is the string index ('0','1','2') and `value` is the element. Confusing them causes incorrect GROUP BY behaviour when trying to aggregate by position.


---
*sql_methods_library - Samantha McGarrigle*